# Defense vs Tech: Correlation and Risk Analysis

This notebook analyzes whether major defense contractors behaved as meaningful diversifiers relative to the technology sector from **2020-01-01 to 2025-12-31**.

Main question:

- did `LMT`, `NOC`, and `GD` provide stable diversification versus `XLK`, or was the relationship regime-dependent?

In [ ]:
# Notebook bootstrap: install anything missing into the current notebook kernel.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "statsmodels": "statsmodels",
    "yfinance": "yfinance",
    "sklearn": "scikit-learn",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages into this notebook kernel:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed in this notebook kernel.")

print("Kernel Python:", sys.executable)

In [ ]:
import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 7)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

TICKERS: List[str] = ["LMT", "NOC", "GD", "XLK"]
DEFENSE_TICKERS: List[str] = ["LMT", "NOC", "GD"]
TECH_TICKER = "XLK"
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"
ROLLING_WINDOW = 30
REGIME_WINDOW = 30
TRADING_DAYS_PER_YEAR = 252
RIDGE_ALPHA = 1.0
LASSO_ALPHA = 0.0005
MOMENTUM_WINDOW = 30

BASE_DIR = Path.cwd()
EXPORT_DIR = BASE_DIR / "outputs"
EXPORT_DIR.mkdir(exist_ok=True)

print(f"Tickers: {TICKERS}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Rolling window: {ROLLING_WINDOW} trading days")

## Data Choices

- Returns and risk metrics use `Adj Close`.
- Missing values are audited first, then forward-filled conservatively.
- Daily frequency is preserved. Smoothing comes from rolling windows rather than resampling.

In [ ]:
# Data loading and preprocessing

def fetch_data(tickers: List[str], start: str, end: str) -> pd.DataFrame:
    df = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        group_by="column",
    )
    if df.empty:
        raise ValueError("No data returned. Check ticker symbols or internet connectivity.")
    return df.sort_index()


def extract_field(raw_df: pd.DataFrame, field: str) -> pd.DataFrame:
    extracted = raw_df[field].copy() if isinstance(raw_df.columns, pd.MultiIndex) else raw_df[[field]].copy()
    if isinstance(extracted, pd.Series):
        extracted = extracted.to_frame(name=TICKERS[0])
    return extracted.sort_index()


def audit_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100,
        "first_valid": df.apply(pd.Series.first_valid_index),
        "last_valid": df.apply(pd.Series.last_valid_index),
    }).sort_values("missing_count", ascending=False)


def clean_price_data(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.sort_index().ffill().dropna(how="all")


def calculate_returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().dropna(how="all")


def normalize_prices(prices: pd.DataFrame, base: float = 100.0) -> pd.DataFrame:
    return prices.div(prices.iloc[0]).mul(base)


raw_data = fetch_data(TICKERS, START_DATE, END_DATE)
adj_close_raw = extract_field(raw_data, "Adj Close")
missing_audit = audit_missing_values(adj_close_raw)
adj_close = clean_price_data(adj_close_raw)
returns = calculate_returns(adj_close)
normalized_prices = normalize_prices(adj_close)

print("Missing-value audit:")
display(missing_audit)
print("Adjusted close sample:")
display(adj_close.head())

## Concepts

- **R-squared (`R^2`)** measures how much of a defense stock's return variation is explained by `XLK`.
- **TSS** means *Total Sum of Squares*: the target series' total variation.
- **RSS** means *Residual Sum of Squares*: the variation left unexplained by the model.
- **Ridge regularization** shrinks coefficients toward zero while usually keeping all features.
- **Lasso regularization** can shrink some coefficients all the way to zero, which acts like feature selection.

In [ ]:
# Statistics, modeling, and compact strategy tests

def annualize_return(daily_returns: pd.Series) -> float:
    compounded = (1 + daily_returns).prod()
    periods = daily_returns.shape[0]
    return compounded ** (TRADING_DAYS_PER_YEAR / periods) - 1 if periods > 0 else np.nan


def annualize_volatility(daily_returns: pd.Series) -> float:
    return daily_returns.std() * np.sqrt(TRADING_DAYS_PER_YEAR)


def calculate_max_drawdown(daily_returns: pd.Series) -> float:
    cumulative = (1 + daily_returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = cumulative.div(running_max) - 1
    return drawdown.min()


def get_summary_stats(returns_df: pd.DataFrame) -> pd.DataFrame:
    rows = {}
    for ticker in returns_df.columns:
        series = returns_df[ticker].dropna()
        rows[ticker] = {
            "annual_return": annualize_return(series),
            "annual_volatility": annualize_volatility(series),
            "max_drawdown": calculate_max_drawdown(series),
            "average_daily_return": series.mean(),
            "daily_volatility": series.std(),
        }
    return pd.DataFrame(rows).T.sort_values("annual_return", ascending=False)


def get_rolling_volatility(returns_df: pd.DataFrame, window: int) -> pd.DataFrame:
    return returns_df.rolling(window).std() * np.sqrt(TRADING_DAYS_PER_YEAR)


def get_correlation_matrix(returns_df: pd.DataFrame) -> pd.DataFrame:
    return returns_df.corr()


def get_rolling_correlation(returns_df: pd.DataFrame, benchmark: str, tickers: List[str], window: int) -> pd.DataFrame:
    data = {}
    for ticker in tickers:
        data[f"{ticker}_vs_{benchmark}"] = returns_df[ticker].rolling(window).corr(returns_df[benchmark])
    return pd.DataFrame(data)


def build_regime_table(rolling_vol_df: pd.DataFrame, returns_df: pd.DataFrame, benchmark: str) -> pd.DataFrame:
    regime_signal = rolling_vol_df[benchmark].dropna()
    quantiles = regime_signal.quantile([0.33, 0.67])
    low_cutoff = quantiles.loc[0.33]
    high_cutoff = quantiles.loc[0.67]
    regime = pd.Series(index=regime_signal.index, dtype="object")
    regime[regime_signal <= low_cutoff] = "Low Vol"
    regime[(regime_signal > low_cutoff) & (regime_signal <= high_cutoff)] = "Medium Vol"
    regime[regime_signal > high_cutoff] = "High Vol"
    regime_returns = returns_df.join(regime.rename("regime"), how="inner")
    grouped = regime_returns.groupby("regime")
    return grouped.mean().T.rename(columns={
        "Low Vol": "avg_return_low_vol",
        "Medium Vol": "avg_return_medium_vol",
        "High Vol": "avg_return_high_vol",
    })


def fit_pair_regression(target: pd.Series, benchmark: pd.Series) -> Dict[str, float]:
    aligned = pd.concat([target, benchmark], axis=1).dropna()
    aligned.columns = ["target", "benchmark"]
    X = sm.add_constant(aligned["benchmark"])
    y = aligned["target"]
    model = sm.OLS(y, X).fit()
    fitted = model.predict(X)
    residuals = y - fitted
    rss = float(np.square(residuals).sum())
    tss = float(np.square(y - y.mean()).sum())
    rmse = float(np.sqrt(mean_squared_error(y, fitted)))
    mae = float(mean_absolute_error(y, fitted))
    return {
        "alpha": float(model.params["const"]),
        "beta": float(model.params["benchmark"]),
        "correlation": float(aligned.corr().iloc[0, 1]),
        "r_squared": float(model.rsquared),
        "rmse": rmse,
        "mae": mae,
        "rss": rss,
        "tss": tss,
        "rss_to_tss": rss / tss if tss else np.nan,
    }


def build_predictive_features(returns_df: pd.DataFrame, target_ticker: str, benchmark_ticker: str) -> pd.DataFrame:
    feature_df = pd.DataFrame(index=returns_df.index)
    feature_df["target_lag_1"] = returns_df[target_ticker].shift(1)
    feature_df["target_lag_5"] = returns_df[target_ticker].rolling(5).mean().shift(1)
    feature_df["benchmark_lag_1"] = returns_df[benchmark_ticker].shift(1)
    feature_df["benchmark_lag_5"] = returns_df[benchmark_ticker].rolling(5).mean().shift(1)
    feature_df["target_next_day"] = returns_df[target_ticker]
    return feature_df.dropna()


def compare_regularized_models(features_df: pd.DataFrame, ridge_alpha: float, lasso_alpha: float) -> pd.DataFrame:
    X = features_df.drop(columns=["target_next_day"])
    y = features_df["target_next_day"]
    split_idx = int(len(features_df) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    models = {
        "OLS": LinearRegression(),
        "Ridge": Ridge(alpha=ridge_alpha),
        "Lasso": Lasso(alpha=lasso_alpha),
    }

    rows = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        rows.append({
            "model": name,
            "test_r_squared": model.score(X_test, y_test),
            "test_rmse": np.sqrt(mean_squared_error(y_test, predictions)),
            "test_mae": mean_absolute_error(y_test, predictions),
        })
    return pd.DataFrame(rows).set_index("model")


def backtest_biased_regime_strategy(returns_df: pd.DataFrame, rolling_vol_df: pd.DataFrame, defense_tickers: List[str], benchmark: str) -> tuple[pd.Series, pd.DataFrame]:
    regime_signal = rolling_vol_df[benchmark].dropna()
    low_cutoff = regime_signal.quantile(0.33)
    high_cutoff = regime_signal.quantile(0.67)
    trailing_return = adj_close[defense_tickers + [benchmark]].pct_change(MOMENTUM_WINDOW)
    trailing_vol = rolling_vol_df[defense_tickers]
    log_rows = []
    strategy_returns = []
    strategy_index = []

    for date in regime_signal.index.intersection(returns_df.index):
        vol_value = regime_signal.loc[date]
        if vol_value <= low_cutoff:
            chosen = benchmark
            label = "Low Vol -> Hold XLK"
        elif vol_value <= high_cutoff:
            chosen = trailing_return.loc[date, defense_tickers].idxmax()
            label = f"Medium Vol -> Momentum {chosen}"
        else:
            chosen = trailing_vol.loc[date, defense_tickers].idxmin()
            label = f"High Vol -> Low-Vol {chosen}"
        strategy_index.append(date)
        strategy_returns.append(returns_df.loc[date, chosen])
        log_rows.append({
            "signal_date": date,
            "chosen_asset": chosen,
            "signal_state": label,
            "strategy_return": returns_df.loc[date, chosen],
            "benchmark_return": returns_df.loc[date, benchmark],
        })

    return pd.Series(strategy_returns, index=strategy_index, name="biased_regime_strategy"), pd.DataFrame(log_rows)


def backtest_regime_strategy(returns_df: pd.DataFrame, rolling_vol_df: pd.DataFrame, defense_tickers: List[str], benchmark: str) -> tuple[pd.Series, pd.DataFrame]:
    regime_signal = rolling_vol_df[benchmark].dropna()
    low_cutoff = regime_signal.quantile(0.33)
    high_cutoff = regime_signal.quantile(0.67)
    trailing_return = adj_close[defense_tickers + [benchmark]].pct_change(MOMENTUM_WINDOW)
    trailing_vol = rolling_vol_df[defense_tickers]
    log_rows = []
    strategy_returns = []
    strategy_index = []

    eligible_dates = regime_signal.index.intersection(returns_df.index)
    for signal_date in eligible_dates[:-1]:
        trade_loc = returns_df.index.get_loc(signal_date) + 1
        trade_date = returns_df.index[trade_loc]
        vol_value = regime_signal.loc[signal_date]
        if vol_value <= low_cutoff:
            chosen = benchmark
            label = "Low Vol -> Hold XLK"
        elif vol_value <= high_cutoff:
            chosen = trailing_return.loc[signal_date, defense_tickers].idxmax()
            label = f"Medium Vol -> Momentum {chosen}"
        else:
            chosen = trailing_vol.loc[signal_date, defense_tickers].idxmin()
            label = f"High Vol -> Low-Vol {chosen}"
        strategy_index.append(trade_date)
        strategy_returns.append(returns_df.loc[trade_date, chosen])
        log_rows.append({
            "signal_date": signal_date,
            "trade_date": trade_date,
            "chosen_asset": chosen,
            "signal_state": label,
            "strategy_return": returns_df.loc[trade_date, chosen],
            "benchmark_return": returns_df.loc[trade_date, benchmark],
        })

    return pd.Series(strategy_returns, index=strategy_index, name="regime_strategy"), pd.DataFrame(log_rows)


def backtest_momentum_strategy(returns_df: pd.DataFrame, rolling_vol_df: pd.DataFrame, defense_tickers: List[str], benchmark: str) -> tuple[pd.Series, pd.DataFrame]:
    benchmark_momentum = adj_close[benchmark].pct_change(MOMENTUM_WINDOW)
    defense_momentum = adj_close[defense_tickers].pct_change(MOMENTUM_WINDOW)
    eligible_dates = returns_df.index.intersection(rolling_vol_df.index)
    log_rows = []
    strategy_returns = []
    strategy_index = []

    for signal_date in eligible_dates[:-1]:
        trade_loc = returns_df.index.get_loc(signal_date) + 1
        trade_date = returns_df.index[trade_loc]
        chosen = benchmark
        label = "Hold XLK"
        if benchmark_momentum.loc[signal_date] < 0:
            qualifying = []
            for ticker in defense_tickers:
                if (
                    defense_momentum.loc[signal_date, ticker] > 0
                    and rolling_vol_df.loc[signal_date, ticker] < rolling_vol_df.loc[signal_date, benchmark]
                ):
                    qualifying.append(ticker)
            if qualifying:
                chosen = defense_momentum.loc[signal_date, qualifying].idxmax()
                label = f"Momentum Filter -> {chosen}"
        strategy_index.append(trade_date)
        strategy_returns.append(returns_df.loc[trade_date, chosen])
        log_rows.append({
            "signal_date": signal_date,
            "trade_date": trade_date,
            "chosen_asset": chosen,
            "signal_state": label,
            "strategy_return": returns_df.loc[trade_date, chosen],
            "benchmark_return": returns_df.loc[trade_date, benchmark],
            "xlk_30d_momentum": benchmark_momentum.loc[signal_date],
        })

    return pd.Series(strategy_returns, index=strategy_index, name="momentum_strategy"), pd.DataFrame(log_rows)


def make_indexed_curve(return_series: pd.Series, base: float = 100.0) -> pd.Series:
    return (1 + return_series.fillna(0)).cumprod() * base


def summarize_strategy(strategy_returns: pd.Series, benchmark_returns: pd.Series) -> pd.DataFrame:
    aligned = pd.concat([strategy_returns, benchmark_returns], axis=1).dropna()
    aligned.columns = ["strategy", "benchmark"]
    rows = {
        "Strategy": {
            "annual_return": annualize_return(aligned["strategy"]),
            "annual_volatility": annualize_volatility(aligned["strategy"]),
            "max_drawdown": calculate_max_drawdown(aligned["strategy"]),
        },
        benchmark_returns.name or "Benchmark": {
            "annual_return": annualize_return(aligned["benchmark"]),
            "annual_volatility": annualize_volatility(aligned["benchmark"]),
            "max_drawdown": calculate_max_drawdown(aligned["benchmark"]),
        },
    }
    return pd.DataFrame(rows).T


summary_stats = get_summary_stats(returns)
rolling_vol = get_rolling_volatility(returns, ROLLING_WINDOW)
correlation_matrix = get_correlation_matrix(returns)
rolling_corr_vs_xlk = get_rolling_correlation(returns, TECH_TICKER, DEFENSE_TICKERS, ROLLING_WINDOW)
regime_table = build_regime_table(rolling_vol, returns, TECH_TICKER)

pair_regression_rows = {}
regularization_rows = []
for ticker in DEFENSE_TICKERS:
    pair_regression_rows[ticker] = fit_pair_regression(returns[ticker], returns[TECH_TICKER])
    feature_df = build_predictive_features(returns, ticker, TECH_TICKER)
    model_table = compare_regularized_models(feature_df, RIDGE_ALPHA, LASSO_ALPHA)
    model_table.insert(0, "ticker", ticker)
    regularization_rows.append(model_table.reset_index())

pair_regression_table = pd.DataFrame(pair_regression_rows).T.sort_values("r_squared", ascending=False)
regularization_results = pd.concat(regularization_rows, ignore_index=True).set_index(["ticker", "model"])

biased_regime_returns, biased_regime_log = backtest_biased_regime_strategy(returns, rolling_vol, DEFENSE_TICKERS, TECH_TICKER)
regime_returns, regime_log = backtest_regime_strategy(returns, rolling_vol, DEFENSE_TICKERS, TECH_TICKER)
momentum_returns, momentum_log = backtest_momentum_strategy(returns, rolling_vol, DEFENSE_TICKERS, TECH_TICKER)

biased_regime_curve = make_indexed_curve(biased_regime_returns)
regime_curve = make_indexed_curve(regime_returns)
momentum_curve = make_indexed_curve(momentum_returns)
xlk_curve_for_biased = make_indexed_curve(returns.loc[biased_regime_returns.index, TECH_TICKER])
xlk_curve_for_regime = make_indexed_curve(returns.loc[regime_returns.index, TECH_TICKER])
xlk_curve_for_momentum = make_indexed_curve(returns.loc[momentum_returns.index, TECH_TICKER])

regime_strategy_summary = summarize_strategy(regime_returns, returns.loc[regime_returns.index, TECH_TICKER].rename("XLK"))
momentum_strategy_summary = summarize_strategy(momentum_returns, returns.loc[momentum_returns.index, TECH_TICKER].rename("XLK"))

print("Summary statistics:")
display(summary_stats)
print("Volatility regime table:")
display(regime_table)
print("Pair regression diagnostics:")
display(pair_regression_table)
print("Regularization comparison:")
display(regularization_results)
print("Regime strategy summary:")
display(regime_strategy_summary)
print("Momentum strategy summary:")
display(momentum_strategy_summary)

## Visualizations

The chart set is intentionally compact: enough to explain the relationship clearly, but still preserving the two strategy experiments you wanted to keep.

In [ ]:
# Visualization helpers

def plot_price_evolution(prices: pd.DataFrame):
    ax = normalize_prices(prices).plot(linewidth=2)
    ax.set_title("Normalized Price Performance (Base = 100)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Indexed Value")
    ax.legend(title="Ticker")
    plt.tight_layout()
    return ax


def plot_rolling_volatility(rolling_vol_df: pd.DataFrame, window: int):
    ax = rolling_vol_df.plot(linewidth=2)
    ax.set_title(f"{window}-Day Rolling Annualized Volatility")
    ax.set_xlabel("Date")
    ax.set_ylabel("Annualized Volatility")
    ax.legend(title="Ticker")
    plt.tight_layout()
    return ax


def plot_correlation_matrix(corr_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr_df, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax)
    ax.set_title("Correlation Matrix of Daily Returns")
    plt.tight_layout()
    return ax


def plot_rolling_correlation(rolling_corr_df: pd.DataFrame, window: int):
    ax = rolling_corr_df.plot(linewidth=2)
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{window}-Day Rolling Correlation vs {TECH_TICKER}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Correlation")
    ax.legend(title="Pair")
    plt.tight_layout()
    return ax


def plot_returns_distribution(returns_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(12, 7))
    for ticker in returns_df.columns:
        sns.kdeplot(returns_df[ticker].dropna(), fill=False, linewidth=2, label=ticker, ax=ax)
    ax.set_title("Daily Return Distributions")
    ax.set_xlabel("Daily Return")
    ax.set_ylabel("Density")
    ax.legend(title="Ticker")
    plt.tight_layout()
    return ax


def plot_strategy_curve(strategy_curve: pd.Series, benchmark_curve: pd.Series, title: str, strategy_label: str):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(strategy_curve.index, strategy_curve.values, label=strategy_label, linewidth=2)
    ax.plot(benchmark_curve.index, benchmark_curve.values, label="XLK", linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Indexed Value")
    ax.legend(title="Series")
    plt.tight_layout()
    return ax

In [ ]:
plot_price_evolution(adj_close)
plt.show()

plot_rolling_volatility(rolling_vol, ROLLING_WINDOW)
plt.show()

plot_correlation_matrix(correlation_matrix)
plt.show()

plot_rolling_correlation(rolling_corr_vs_xlk, ROLLING_WINDOW)
plt.show()

plot_returns_distribution(returns)
plt.show()

### Chart Notes

- **Normalized price performance:** compares cumulative performance from a common starting value.
- **Rolling volatility:** shows how risk changed over time rather than treating volatility as fixed.
- **Correlation matrix:** shows how closely defense names moved with one another and with `XLK`.
- **Rolling correlation vs XLK:** shows whether diversification benefits were stable or regime-dependent.
- **Return distributions:** show dispersion and tail behavior beyond average returns.

## Strategy Section 1: Regime Strategy

This section keeps both versions of the regime strategy:

- the original biased version, which uses same-day information and therefore overstates performance
- the corrected lagged version, which waits one trading day before acting on the signal

In [ ]:
plot_strategy_curve(
    biased_regime_curve,
    xlk_curve_for_biased,
    "Biased Regime Strategy vs XLK (False Positive Example)",
    "Biased Regime Strategy",
)
plt.show()

plot_strategy_curve(
    regime_curve,
    xlk_curve_for_regime,
    "Lagged Regime Strategy vs XLK",
    "Lagged Regime Strategy",
)
plt.show()

display(regime_strategy_summary)
display(regime_log.head(10))

## Strategy Section 2: Momentum With Defense Filter

Rule summary:

- stay in `XLK` by default
- only rotate when `XLK` 30-day momentum is negative
- require a defense stock to have positive 30-day momentum and lower rolling volatility than `XLK`
- if multiple names qualify, choose the strongest-momentum defense stock

In [ ]:
plot_strategy_curve(
    momentum_curve,
    xlk_curve_for_momentum,
    "Momentum With Defense Filter Strategy vs XLK",
    "Momentum Filter Strategy",
)
plt.show()

display(momentum_strategy_summary)
display(momentum_log.head(10))

In [ ]:
# Insight generation
best_return = summary_stats["annual_return"].idxmax()
lowest_vol = summary_stats["annual_volatility"].idxmin()
weakest_corr = correlation_matrix.loc[DEFENSE_TICKERS, TECH_TICKER].idxmin()
strongest_fit = pair_regression_table["r_squared"].idxmax()

print("Key takeaways:")
print(f"1. {best_return} had the highest annualized return over the full sample.")
print(f"2. {lowest_vol} had the lowest annualized volatility over the sample.")
print(f"3. {weakest_corr} had the weakest full-period return correlation to {TECH_TICKER}, making it the best candidate for conditional diversification.")
print(f"4. {strongest_fit} had the strongest regression fit versus {TECH_TICKER}, meaning it moved most like tech among the defense names.")
print("5. The strategy tests are useful mainly as diagnostics: they show how easy it is to overstate signal quality and how difficult it was to beat a strong XLK trend with simple defense overlays.")

## Final Reflection Prompts

1. Which defense stock looked most independent from `XLK` over time?
2. Did lower correlation also coincide with lower volatility or better drawdown behavior?
3. Which defense stock behaved most like a tech-sensitive equity rather than a defensive diversifier?
4. Did the regularized models materially outperform plain OLS on the next-day prediction task, or was predictive signal weak?
5. Why did the biased regime strategy look much stronger than the corrected lagged version?
6. Did the momentum filter improve practical timing, or mostly reveal how dominant `XLK` remained in this sample?